In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine learning libraries
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, 
                            classification_report, confusion_matrix)

# Import TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input, Conv1D, MaxPooling1D, Flatten, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
from tkinter import font as tkfont
import os

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ================= DATA LOADING AND PREPROCESSING =================
try:
    df = pd.read_csv("data.csv")
    print("Successfully loaded data.csv")
except Exception as e:
    print(f"Error loading data.csv: {e}")
    exit(1)

# Display dataset info
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Handle missing values and drop irrelevant columns
df = df.dropna()
if "profile_id" in df.columns:
    df = df.drop(columns=["profile_id"])

# Convert numeric columns to float32 for efficiency
num_cols = ['u_q', 'coolant', 'stator_winding', 'u_d', 'stator_tooth', 
           'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque']
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].astype('float32')

# ================= FEATURE SELECTION AND DATA SPLITTING =================
# Select features and target variables
features = [col for col in num_cols if col != 'stator_winding' and col in df.columns]
target_reg = 'stator_winding'
X = df[features]
y_reg = df[target_reg].astype('float32')

# Create binary classification target (high/low temperature)
df['high_temp'] = (df[target_reg] > df[target_reg].median()).astype('int32')
y_cls = df['high_temp']

# Split data into training and testing sets
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X, y_cls, test_size=0.2, random_state=42)

# Scale features
scaler_reg = MinMaxScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

scaler_cls = MinMaxScaler()
X_train_cls_scaled = scaler_cls.fit_transform(X_train_cls)
X_test_cls_scaled = scaler_cls.transform(X_test_cls)

# Reshape data for 1D CNN
X_train_reg_cnn = X_train_reg_scaled.reshape(X_train_reg_scaled.shape[0], X_train_reg_scaled.shape[1], 1)
X_test_reg_cnn = X_test_reg_scaled.reshape(X_test_reg_scaled.shape[0], X_test_reg_scaled.shape[1], 1)

X_train_cls_cnn = X_train_cls_scaled.reshape(X_train_cls_scaled.shape[0], X_train_cls_scaled.shape[1], 1)
X_test_cls_cnn = X_test_cls_scaled.reshape(X_test_cls_scaled.shape[0], X_test_cls_scaled.shape[1], 1)

# Convert classification labels to categorical
y_train_cls_cat = to_categorical(y_train_cls)
y_test_cls_cat = to_categorical(y_test_cls)

print("Data preparation complete.")

In [ ]:
# ================= MODEL CALLBACKS =================
# Define callbacks for training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, min_lr=1e-6)
]

# ================= SIMPLE MLP FOR REGRESSION =================
def build_simple_mlp_reg(input_dim):
    model = Sequential([
        Dense(64, activation='relu', input_dim=input_dim, kernel_regularizer=l2(0.001)),
        Dropout(0.2),
        Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.2),
        Dense(1)  # Output layer for regression
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Build and train simple MLP for regression
print("\n===== Training Simple MLP for Regression =====")
input_dim_reg = X_train_reg_scaled.shape[1]
simple_mlp_reg = build_simple_mlp_reg(input_dim_reg)
simple_mlp_reg.summary()

history_simple_reg = simple_mlp_reg.fit(
    X_train_reg_scaled, y_train_reg,
    validation_split=0.2,
    epochs=50,  # Reduced epochs
    batch_size=1024,  # Smaller batch size for CPU
    callbacks=callbacks,
    verbose=1
)

# Evaluate regression model
y_pred_simple_reg = simple_mlp_reg.predict(X_test_reg_scaled)
r2_simple = r2_score(y_test_reg, y_pred_simple_reg)
mse_simple = mean_squared_error(y_test_reg, y_pred_simple_reg)
print(f"Simple MLP - Test R² Score: {r2_simple:.4f}")
print(f"Simple MLP - Test MSE: {mse_simple:.4f}")

# Plot regression results
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_test_reg, y_pred_simple_reg, alpha=0.5)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title('Simple MLP: Predictions vs Actual')

plt.subplot(1, 2, 2)
plt.plot(history_simple_reg.history['loss'], label='Training Loss')
plt.plot(history_simple_reg.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Simple MLP: Training History')
plt.tight_layout()
plt.savefig('simple_mlp_reg_results.png')
plt.close()

# ================= SIMPLE MLP FOR CLASSIFICATION =================
def build_simple_mlp_cls(input_dim, num_classes):
    model = Sequential([
        Dense(64, activation='relu', input_dim=input_dim, kernel_regularizer=l2(0.001)),
        Dropout(0.2),
        Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')  # Output layer for classification
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Build and train simple MLP for classification
print("\n===== Training Simple MLP for Classification =====")
input_dim_cls = X_train_cls_scaled.shape[1]
num_classes = len(np.unique(y_train_cls))
simple_mlp_cls = build_simple_mlp_cls(input_dim_cls, num_classes)
simple_mlp_cls.summary()

history_simple_cls = simple_mlp_cls.fit(
    X_train_cls_scaled, y_train_cls_cat,
    validation_split=0.2,
    epochs=50,  # Reduced epochs
    batch_size=1024,  # Smaller batch size for CPU
    callbacks=callbacks,
    verbose=1
)

# Evaluate classification model
y_pred_simple_cls_prob = simple_mlp_cls.predict(X_test_cls_scaled)
y_pred_simple_cls = np.argmax(y_pred_simple_cls_prob, axis=1)
accuracy_simple = accuracy_score(y_test_cls, y_pred_simple_cls)
print(f"Simple MLP - Test Accuracy: {accuracy_simple:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_cls, y_pred_simple_cls))

# Plot classification results
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test_cls, y_pred_simple_cls)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal Temp', 'High Temp'], 
            yticklabels=['Normal Temp', 'High Temp'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Simple MLP: Confusion Matrix')

plt.subplot(1, 2, 2)
plt.plot(history_simple_cls.history['accuracy'], label='Training Accuracy')
plt.plot(history_simple_cls.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Simple MLP: Training History')
plt.tight_layout()
plt.savefig('simple_mlp_cls_results.png')
plt.close()


In [ ]:
# ================= 1D CNN FOR REGRESSION =================
def build_cnn_reg(input_shape):
    model = Sequential([
        Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape, padding='same'),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.2),
        Dense(1)  # Output layer for regression
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Build and train 1D CNN for regression
print("\n===== Training 1D CNN for Regression =====")
input_shape_reg = (X_train_reg_scaled.shape[1], 1)
cnn_reg = build_cnn_reg(input_shape_reg)
cnn_reg.summary()

history_cnn_reg = cnn_reg.fit(
    X_train_reg_cnn, y_train_reg,
    validation_split=0.2,
    epochs=50,  # Reduced epochs
    batch_size=1024,  # Smaller batch size for CPU
    callbacks=callbacks,
    verbose=1
)

# Evaluate CNN regression model
y_pred_cnn_reg = cnn_reg.predict(X_test_reg_cnn)
r2_cnn = r2_score(y_test_reg, y_pred_cnn_reg)
mse_cnn = mean_squared_error(y_test_reg, y_pred_cnn_reg)
print(f"1D CNN - Test R² Score: {r2_cnn:.4f}")
print(f"1D CNN - Test MSE: {mse_cnn:.4f}")

# Plot CNN regression results
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_test_reg, y_pred_cnn_reg, alpha=0.5)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title('1D CNN: Predictions vs Actual')

plt.subplot(1, 2, 2)
plt.plot(history_cnn_reg.history['loss'], label='Training Loss')
plt.plot(history_cnn_reg.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('1D CNN: Training History')
plt.tight_layout()
plt.savefig('cnn_reg_results.png')
plt.close()

In [ ]:
# ================= AUTOENCODER + MLP FOR REGRESSION =================
def build_autoencoder_mlp_reg(input_dim):
    # Define encoder
    encoding_dim = 6  # Reduced bottleneck size
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(16, activation='relu', kernel_regularizer=l2(0.001))(input_layer)
    bottleneck = Dense(encoding_dim, activation='relu', name='bottleneck')(encoded)
    
    # Define decoder for pretraining
    decoded = Dense(16, activation='relu')(bottleneck)
    decoded = Dense(input_dim, activation='sigmoid')(decoded)
    
    # Autoencoder model (encoder + decoder)
    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    
    # Pretrain autoencoder
    print("Pre-training autoencoder...")
    autoencoder.fit(
        X_train_reg_scaled, X_train_reg_scaled,  # Train to reconstruct inputs
        validation_split=0.2,
        epochs=25,  # Reduced epochs for pretraining
        batch_size=1024,
        callbacks=callbacks,
        verbose=1
    )
    
    # Extract encoder part
    encoder = Model(inputs=input_layer, outputs=bottleneck)
    
    # Build MLP on top of encoder
    encoded_features = encoder(input_layer)
    x = Dense(16, activation='relu', kernel_regularizer=l2(0.001))(encoded_features)
    x = Dropout(0.2)(x)
    output = Dense(1)(x)  # Regression output
    
    # Combined model
    ae_mlp = Model(inputs=input_layer, outputs=output)
    ae_mlp.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    
    return autoencoder, encoder, ae_mlp

# Build and train autoencoder + MLP for regression
print("\n===== Training Autoencoder + MLP for Regression =====")
_, _, ae_mlp_reg = build_autoencoder_mlp_reg(input_dim_reg)
ae_mlp_reg.summary()

history_ae_mlp_reg = ae_mlp_reg.fit(
    X_train_reg_scaled, y_train_reg,
    validation_split=0.2,
    epochs=50,  # Reduced epochs
    batch_size=1024,  # Smaller batch size for CPU
    callbacks=callbacks,
    verbose=1
)

# Evaluate autoencoder + MLP regression model
y_pred_ae_mlp_reg = ae_mlp_reg.predict(X_test_reg_scaled)
r2_ae_mlp = r2_score(y_test_reg, y_pred_ae_mlp_reg)
mse_ae_mlp = mean_squared_error(y_test_reg, y_pred_ae_mlp_reg)
print(f"Autoencoder + MLP - Test R² Score: {r2_ae_mlp:.4f}")
print(f"Autoencoder + MLP - Test MSE: {mse_ae_mlp:.4f}")

# Plot autoencoder + MLP regression results
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_test_reg, y_pred_ae_mlp_reg, alpha=0.5)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title('Autoencoder + MLP: Predictions vs Actual')

plt.subplot(1, 2, 2)
plt.plot(history_ae_mlp_reg.history['loss'], label='Training Loss')
plt.plot(history_ae_mlp_reg.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Autoencoder + MLP: Training History')
plt.tight_layout()
plt.savefig('ae_mlp_reg_results.png')
plt.close()


In [ ]:
# ================= DEEP MLP FOR REGRESSION =================
def build_deep_mlp_reg(input_dim):
    model = Sequential([
        # Input layer
        Dense(128, activation='relu', input_dim=input_dim, kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.2),
        
        # Hidden layers - making it deeper with decreasing neuron counts
        Dense(96, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.2),
        
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.2),
        
        # Output layer for regression
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Build and train deep MLP for regression
print("\n===== Training Deep MLP for Regression =====")
deep_mlp_reg = build_deep_mlp_reg(input_dim_reg)
deep_mlp_reg.summary()

# Modified callbacks for deep model to prevent overfitting
deep_callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=7, min_lr=1e-6)
]

history_deep_reg = deep_mlp_reg.fit(
    X_train_reg_scaled, y_train_reg,
    validation_split=0.2,
    epochs=100,  # More epochs but with early stopping
    batch_size=1024,
    callbacks=deep_callbacks,
    verbose=1
)

# Evaluate deep MLP regression model
y_pred_deep_reg = deep_mlp_reg.predict(X_test_reg_scaled)
r2_deep = r2_score(y_test_reg, y_pred_deep_reg)
mse_deep = mean_squared_error(y_test_reg, y_pred_deep_reg)
print(f"Deep MLP - Test R² Score: {r2_deep:.4f}")
print(f"Deep MLP - Test MSE: {mse_deep:.4f}")

# Plot deep MLP regression results
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_test_reg, y_pred_deep_reg, alpha=0.5)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.title('Deep MLP: Predictions vs Actual')

plt.subplot(1, 2, 2)
plt.plot(history_deep_reg.history['loss'], label='Training Loss')
plt.plot(history_deep_reg.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Deep MLP: Training History')
plt.tight_layout()
plt.savefig('deep_mlp_reg_results.png')
plt.close()

# ================= DEEP MLP FOR CLASSIFICATION =================
def build_deep_mlp_cls(input_dim, num_classes):
    model = Sequential([
        # Input layer
        Dense(128, activation='relu', input_dim=input_dim, kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.2),
        
        # Hidden layers
        Dense(96, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.2),
        
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.2),
        
        # Output layer for classification
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Build and train deep MLP for classification
print("\n===== Training Deep MLP for Classification =====")
deep_mlp_cls = build_deep_mlp_cls(input_dim_cls, num_classes)
deep_mlp_cls.summary()

history_deep_cls = deep_mlp_cls.fit(
    X_train_cls_scaled, y_train_cls_cat,
    validation_split=0.2,
    epochs=100,  # More epochs but with early stopping
    batch_size=1024,
    callbacks=deep_callbacks,
    verbose=1
)

# Evaluate deep MLP classification model
y_pred_deep_cls_prob = deep_mlp_cls.predict(X_test_cls_scaled)
y_pred_deep_cls = np.argmax(y_pred_deep_cls_prob, axis=1)
accuracy_deep = accuracy_score(y_test_cls, y_pred_deep_cls)
print(f"Deep MLP - Test Accuracy: {accuracy_deep:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_cls, y_pred_deep_cls))

# Plot deep MLP classification results
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test_cls, y_pred_deep_cls)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal Temp', 'High Temp'], 
            yticklabels=['Normal Temp', 'High Temp'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Deep MLP: Confusion Matrix')

plt.subplot(1, 2, 2)
plt.plot(history_deep_cls.history['accuracy'], label='Training Accuracy')
plt.plot(history_deep_cls.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Deep MLP: Training History')
plt.tight_layout()
plt.savefig('deep_mlp_cls_results.png')
plt.close()

In [ ]:

# ================= UPDATE MODEL COMPARISON =================
# Update regression model comparison
reg_models = ['Simple MLP', '1D CNN', 'Autoencoder + MLP', 'Deep MLP']
r2_scores = [r2_simple, r2_cnn, r2_ae_mlp, r2_deep]
mse_scores = [mse_simple, mse_cnn, mse_ae_mlp, mse_deep]

plt.figure(figsize=(14, 5))
# Plotting code for bar charts...

plt.subplot(1, 2, 1)
bars = plt.bar(reg_models, r2_scores, color=['skyblue', 'lightgreen', 'gold', 'salmon'])
plt.ylabel('R² Score (higher is better)')
plt.title('Model Comparison: R² Score')
for bar, score in zip(bars, r2_scores):
    plt.text(bar.get_x() + bar.get_width()/2, score + 0.01, f'{score:.4f}', 
             ha='center', va='bottom')

plt.subplot(1, 2, 2)
bars = plt.bar(reg_models, mse_scores, color=['skyblue', 'lightgreen', 'gold', 'salmon'])
plt.ylabel('MSE (lower is better)')
plt.title('Model Comparison: MSE')
for bar, score in zip(bars, mse_scores):
    plt.text(bar.get_x() + bar.get_width()/2, score + 0.01, f'{score:.4f}', 
             ha='center', va='bottom')

plt.tight_layout()
plt.savefig('updated_model_comparison.png')
plt.close()

# Classification models comparison
cls_models = ['Simple MLP', 'Deep MLP']
acc_scores = [accuracy_simple, accuracy_deep]

plt.figure(figsize=(8, 5))
bars = plt.bar(cls_models, acc_scores, color=['skyblue', 'salmon'])
plt.ylabel('Accuracy (higher is better)')
plt.title('Classification Model Comparison: Accuracy')
for bar, score in zip(bars, acc_scores):
    plt.text(bar.get_x() + bar.get_width()/2, score + 0.01, f'{score:.4f}', 
             ha='center', va='bottom')

plt.tight_layout()
plt.savefig('classification_model_comparison.png')
plt.close()

print("\nDeep MLP models trained and evaluated. Updated comparison results saved as images.")

In [ ]:
class ModelResultsViewer:
    def __init__(self, root):
        self.root = root
        self.root.title("Deep Learning Model Results")
        
        # Set fixed window size
        window_width = 1000
        window_height = 700
        self.root.geometry(f"{window_width}x{window_height}")
        self.root.resizable(False, False)
        
        # Define color scheme - Dark blue theme with modern accents
        self.colors = {
            "bg_dark": "#10182c",           # Dark blue background
            "bg_medium": "#1a2940",         # Medium blue for frames
            "bg_light": "#253656",          # Light blue for buttons
            "accent": "#4c9aff",            # Accent blue for highlights
            "text_primary": "#ffffff",      # White text
            "text_secondary": "#c2c8d5",    # Light gray text
            "button_hover": "#1d3da9",      # Button hover color
            "button_pressed": "#2d5ca0"     # Button pressed color
        }
        
        # Apply dark theme to root window
        self.root.configure(bg=self.colors["bg_dark"])
        
        # Configure custom styles
        self.setup_styles()
        
        # Main frame
        main_frame = ttk.Frame(root, style="Main.TFrame", padding=(20, 20, 20, 20))
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Header
        header_label = ttk.Label(main_frame, text="Deep Learning Model Results Viewer", 
                                style="Topper.TLabel")
        header_label.pack(fill=tk.Y, pady=(0, 20))
        
        # Create frame for buttons and image display
        content_frame = ttk.Frame(main_frame, style="Content.TFrame")
        content_frame.pack(fill=tk.BOTH, expand=True)
        
        # Button frame on the left
        button_frame = ttk.Frame(content_frame, style="Button.TFrame", width=250)
        button_frame.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 20))
        
        # Image display frame on the right
        self.image_frame = ttk.Frame(content_frame, style="Image.TFrame")
        self.image_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        # Image display label
        self.image_label = ttk.Label(self.image_frame, style="Image.TLabel")
        self.image_label.pack(fill=tk.BOTH, expand=True)
        
        # Current image info label
        self.info_label = ttk.Label(self.image_frame, text="No image selected", 
                                   style="Info.TLabel", anchor="center")
        self.info_label.pack(pady=(10, 0))
        
        # Button styling
        button_width = 28
        button_pad = 8
        
        # Group 1: Regression Models
        group1_label = ttk.Label(button_frame, text="Regression Models", style="Header.TLabel")
        group1_label.pack(anchor="w", pady=(0, 5))
        
        # Regression model buttons - using custom Button class instead of ttk.Button
        reg_buttons = [
            ("Simple MLP Regression", "simple_mlp_reg_results.png"),
            ("1D CNN Regression", "cnn_reg_results.png"),
            ("Autoencoder + MLP", "ae_mlp_reg_results.png"),
            ("Deep MLP Regression", "deep_mlp_reg_results.png")
        ]
        
        for text, img_path in reg_buttons:
            btn = self.create_custom_button(button_frame, text, img_path)
            btn.pack(pady=button_pad, fill=tk.X)
        
        # Group 2: Classification Models
        group2_label = ttk.Label(button_frame, text="Classification Models", 
                               style="Header.TLabel")
        group2_label.pack(anchor="w", pady=(20, 5))
        
        # Classification model buttons
        cls_buttons = [
            ("Simple MLP Classification", "simple_mlp_cls_results.png"),
            ("Deep MLP Classification", "deep_mlp_cls_results.png")
        ]
        
        for text, img_path in cls_buttons:
            btn = self.create_custom_button(button_frame, text, img_path)
            btn.pack(pady=button_pad, fill=tk.X)
        
        # Group 3: Comparison Results
        group3_label = ttk.Label(button_frame, text="Comparison Results", 
                               style="Header.TLabel")
        group3_label.pack(anchor="w", pady=(20, 5))
        
        # Comparison buttons
        comp_buttons = [
            ("Regression Model Comparison", "updated_model_comparison.png"),
            ("Classification Model Comparison", "classification_model_comparison.png")
        ]
        
        for text, img_path in comp_buttons:
            btn = self.create_custom_button(button_frame, text, img_path)
            btn.pack(pady=button_pad, fill=tk.X)
            
    def create_custom_button(self, parent, text, img_path):
        """Create a custom tk Button with the right styling instead of ttk.Button"""
        btn = tk.Button(
            parent,
            text=text,
            font=(self.font_family, 10),
            bg=self.colors["bg_dark"],
            fg=self.colors["text_primary"],
            activebackground=self.colors["button_hover"],
            activeforeground=self.colors["text_primary"],
            borderwidth=4,
            relief=tk.FLAT,
            highlightcolor="orange",
            highlightbackground=self.colors["button_hover"],
            highlightthickness=1,
            pady=1,
            cursor="hand2",
            width=28,
            command=lambda p=img_path, t=text: self.show_image(p, t)
        )
        
        # Add hover effects
        btn.bind("<Enter>", lambda e, b=btn: self.on_button_hover(e, b))
        btn.bind("<Leave>", lambda e, b=btn: self.on_button_leave(e, b))
        
        return btn
        
    def setup_styles(self):
        """Configure custom styles for the dark theme"""
        self.style = ttk.Style()
        
        # Import a modern font if available, fallback to system fonts
        available_fonts = tkfont.families()
        modern_fonts = ["Rubik", "Montserrat", "Roboto", "Segoe UI", "Helvetica Neue", "Arial"]
        
        # Find the first available modern font
        self.font_family = next((font for font in modern_fonts if font in available_fonts), "Arial")
        
        # Configure frame styles
        self.style.configure("Main.TFrame", background=self.colors["bg_dark"])
        self.style.configure("Content.TFrame", background=self.colors["bg_dark"])
        self.style.configure("Button.TFrame", background=self.colors["bg_dark"])
        self.style.configure("Image.TFrame", background=self.colors["bg_dark"])
        
        # Configure label styles
        self.style.configure("TLabel", 
                           background=self.colors["bg_dark"],
                           foreground=self.colors["text_secondary"],
                           font=(self.font_family, 12,"bold"))
        
        self.style.configure("Header.TLabel", 
                           background=self.colors["bg_dark"],
                           foreground=self.colors["accent"],
                           font=(self.font_family, 14, "bold"))
        
        self.style.configure("Topper.TLabel", 
                           background=self.colors["bg_dark"],
                           foreground=self.colors["text_primary"],
                           font=(self.font_family, 22, "bold"))
        
        self.style.configure("Info.TLabel", 
                           background=self.colors["bg_dark"],
                           foreground=self.colors["accent"],
                           font=(self.font_family, 12, "bold"))
        
        self.style.configure("Image.TLabel", 
                           background=self.colors["bg_dark"])

    def on_button_hover(self, event, button):
        """Change button color on hover"""
        button.config(
            background=self.colors["button_hover"],
            foreground=self.colors["text_primary"],
            font=(self.font_family, 10, "bold")
        )

    def on_button_leave(self, event, button):
        """Reset button color when mouse leaves"""
        button.config(
            background=self.colors["bg_dark"],
            foreground=self.colors["text_primary"],
            font=(self.font_family, 10)
        )

    def show_image(self, image_path, title):
        """Display the selected image"""
        try:
            # Check if image exists
            if not os.path.exists(image_path):
                self.info_label.config(text=f"Image not found: {image_path}")
                # Clear image
                self.image_label.config(image="")
                return
                
            # Open and resize image
            img = Image.open(image_path)
            
            # Calculate proper resize to fit the frame while maintaining aspect ratio
            img_width, img_height = img.size
            frame_width = self.image_frame.winfo_width() - 20  # padding
            frame_height = self.image_frame.winfo_height() - 50  # space for label
            
            # Only resize if frame is properly initialized
            if frame_width > 1 and frame_height > 1:
                # Calculate scale factor to fit in frame
                width_ratio = frame_width / img_width
                height_ratio = frame_height / img_height
                scale_factor = min(width_ratio, height_ratio)
                
                # Resize image
                new_width = int(img_width * scale_factor)
                new_height = int(img_height * scale_factor)
                img = img.resize((new_width, new_height), Image.LANCZOS)
            
            # Convert to PhotoImage and display
            photo = ImageTk.PhotoImage(img)
            self.image_label.config(image=photo)
            self.image_label.image = photo  # Keep a reference to prevent garbage collection
            
            # Update info label
            self.info_label.config(text=f"{title}")
            
        except Exception as e:
            self.info_label.config(text=f"Error loading image: {str(e)}")

def start_gui():
    
    # Check if running inside notebook
    try:
        import IPython
        from IPython.display import display
        is_notebook = True
    except ImportError:
        is_notebook = False
    
    if is_notebook:
        from tkinter import Tk
        import threading
        import time
        
        # Function to run GUI in separate thread
        def run_gui():
            root = Tk()
            app = ModelResultsViewer(root)
            
            # Wait until window is ready before showing images
            root.update_idletasks()
            
            # Show first image by default
            time.sleep(0.5)  # Give UI time to render
            app.show_image("place.png", "Welcome to Model Results Viewer")
            
            root.mainloop()
        
        # Start GUI in a new thread
        thread = threading.Thread(target=run_gui)
        thread.daemon = True
        thread.start()
        
        print("GUI started in a separate window. Close the window when done.")
    else:
        # Regular execution outside notebook
        root = tk.Tk()
        app = ModelResultsViewer(root)
        root.after(500, lambda: app.show_image("place.png", "Welcome to Model Results Viewer"))
        root.mainloop()

# Call this function to start the GUI
if __name__ == "__main__" or 'get_ipython' in globals():
    start_gui()